# Proyecto 3 - Incertidumbre
## Clasificador SPAM/HAM con Bayes

Este notebook completa el proyecto a partir del laboratorio 6: EDA, limpieza, modelo bayesiano, pruebas de rendimiento y modulo de prediccion para presentacion.

## 1. Carga y configuracion

Se carga el dataset `spam_ham.csv`, se normalizan las etiquetas y se preparan librerias para analisis y modelado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from spam_ham_bayes_project import (
    SpamHamBayesClassifier,
    evaluate_thresholds,
    load_dataset,
    train_project_model,
)

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")

df = load_dataset("spam_ham.csv")
print("Dimensiones:", df.shape)
df.head()

## 2. Analisis exploratorio de datos

El EDA permite entender el desbalance de clases, las longitudes y el vocabulario que diferencia spam de ham.

In [ ]:
class_counts = df["label"].value_counts()
class_props = df["label"].value_counts(normalize=True) * 100

print("Cantidad por clase:")
print(class_counts)
print("\nProporcion por clase (%):")
print(class_props.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
class_counts.plot(kind="bar", ax=axes[0], color=["#4c78a8", "#f58518"])
axes[0].set_title("Distribucion spam vs ham")
axes[0].set_xlabel("Clase")
axes[0].set_ylabel("Mensajes")
axes[0].tick_params(axis="x", rotation=0)
class_props.plot(kind="pie", autopct="%1.1f%%", startangle=90, ax=axes[1])
axes[1].set_title("Proporcion por clase")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
df["raw_length_chars"] = df["message"].str.len()
df["raw_word_count"] = df["message"].str.split().str.len()

summary_raw = df.groupby("label")[["raw_length_chars", "raw_word_count"]].agg(["mean", "median", "max"]).round(2)
summary_raw

In [ ]:
sns.kdeplot(data=df, x="raw_length_chars", hue="label", fill=True, common_norm=False, alpha=0.35)
plt.title("Densidad de longitud antes del preprocesamiento")
plt.xlabel("Caracteres")
plt.ylabel("Densidad")
plt.show()

## 3. Limpieza de datos

La limpieza incluye minusculas, tokenizacion, eliminacion de tokens no alfabeticos, stopwords y stemming. Esto reduce ruido y deja palabras mas comparables.

In [ ]:
model_for_cleaning = SpamHamBayesClassifier()
df["clean_tokens"] = df["message"].apply(model_for_cleaning.preprocess)
df["clean_message"] = df["clean_tokens"].apply(lambda tokens: " ".join(tokens))
df["clean_length_chars"] = df["clean_message"].str.len()
df["clean_word_count"] = df["clean_tokens"].str.len()

df[["label", "message", "clean_tokens", "clean_message"]].head()

In [ ]:
summary_clean = df.groupby("label")[["raw_length_chars", "raw_word_count", "clean_length_chars", "clean_word_count"]].mean().round(2)
summary_clean

In [ ]:
def top_words(dataframe, label, token_column="clean_tokens", top_n=20):
    counter = Counter()
    for tokens in dataframe.loc[dataframe["label"] == label, token_column]:
        counter.update(tokens)
    return pd.DataFrame(counter.most_common(top_n), columns=["word", "frequency"])

top_spam = top_words(df, "spam")
top_ham = top_words(df, "ham")
display(top_spam)
display(top_ham)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, title in [(axes[0], top_spam, "Top palabras spam"), (axes[1], top_ham, "Top palabras ham")]:
    ordered = data.sort_values("frequency")
    ax.barh(ordered["word"], ordered["frequency"], color="#4c78a8")
    ax.set_title(title)
    ax.set_xlabel("Frecuencia")
plt.tight_layout()
plt.show()

## 4. Modelo bayesiano

Para cada palabra W se calcula `P(S|W)` usando la formula del anexo. Se usa presencia por documento y suavizado de Laplace para evitar probabilidades cero. Luego se combinan las probabilidades individuales de las palabras del texto.

In [ ]:
model, train_df, test_df = train_project_model("spam_ham.csv", random_state=42)

print("Entrenamiento:", train_df.shape)
print("Prueba:", test_df.shape)
print(f"P(spam) = {model.prior_spam:.4f}")
print(f"P(ham) = {model.prior_ham:.4f}")
print("Vocabulario:", len(model.vocabulary))

In [ ]:
predictive_words = (
    pd.DataFrame(
        [(word, probability) for word, probability in model.word_spam_probability.items()],
        columns=["word", "p_spam_given_word"],
    )
    .sort_values("p_spam_given_word", ascending=False)
    .head(20)
)
predictive_words

In [ ]:
plt.figure(figsize=(10, 6))
ordered = predictive_words.sort_values("p_spam_given_word")
plt.barh(ordered["word"], ordered["p_spam_given_word"], color="#f58518")
plt.title("Palabras con mayor probabilidad de spam")
plt.xlabel("P(spam | palabra)")
plt.show()

## 5. Pruebas de rendimiento

Se evalua sobre el 20% de prueba y se exploran thresholds de clasificacion entre 0.1 y 0.9.

In [ ]:
metrics = evaluate_thresholds(model, test_df)
metrics.round(4)

In [ ]:
best_row = metrics.loc[metrics["f1_score"].idxmax()]
print("Mejor threshold por F1:")
print(best_row.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
for column in ["precision", "recall", "f1_score"]:
    ax.plot(metrics["threshold"], metrics[column], marker="o", label=column)
ax.set_title("Metricas por threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Valor")
ax.legend()
plt.show()

## 6. Modulo para presentacion

La funcion `classify_prompt` acepta texto libre y retorna probabilidad de spam, prediccion y las 3 palabras con mayor poder predictivo dentro del prompt.

In [ ]:
prompt = "Free entry to win a cash prize. Claim now by calling this number."
model.classify_prompt(prompt, threshold=float(best_row["threshold"]))

In [ ]:
def clasificar_sms(texto):
    return model.classify_prompt(texto, threshold=float(best_row["threshold"]))

# Para la presentacion, cambie el texto de ejemplo por cualquier SMS.
clasificar_sms("Congratulations, you won a prize. Claim your free reward now.")

## 7. Discusion

El threshold 0.6 obtiene el mejor F1 en esta corrida. El modelo alcanza precision alta, lo que significa que casi todo lo marcado como spam realmente es spam. El recall es menor, por lo que algunos spam se clasifican como ham. Esto refleja una decision conservadora: se minimizan falsos positivos, algo importante si no se quiere bloquear mensajes legitimos.

La limpieza mejora la interpretabilidad y reduce ruido, pero eliminar numeros puede quitar senales utiles como telefonos, codigos y enlaces. Como mejora futura se podrian agregar caracteristicas binarias para URL, numeros, mayusculas, signos de exclamacion y longitud del mensaje.